## Agent 的高级用法-流式输出

### 1. values 输出模式

当 stream_mode 设置为 values 模式时, 每个步骤执行后, 都会输出完整的状态信息, 适用于每一步都要获取完整状态, 状态持久化场景

In [25]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
import os
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
from rich import print as rprint
from pydantic import BaseModel, Field
from typing import Union, Dict, Any

load_dotenv(override=True)
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_BASE_URL = os.getenv("ZHIPUAI_API_BASE_URL")

# 1. 模型初始化
model = init_chat_model(
    model='glm-5.3',
    model_provider="openai",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_API_BASE_URL
)

In [ ]:
@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息

    Args:
        customer_id: 客户 ID, 用户唯一标识客户

    Returns:
        包含客户基本信息的字典, 如姓名, 等级, 加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三", "level": "VIP", "join_date": "2023-01-15"}

@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史
    
    Args:
        customer_id: 客户ID, 用于唯一标识客户

    Returns:
        包含客户订单历史的字典, 如总订单数, 总花费等
    """
    return {"total_orders": 15, "total_spent": 25800.00}

@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动

    Returns:
        包含当前可用促销活动的字典, 如活动名称, 有效日期等
    """
    return {
        "promotions": ["老用户优惠", "会员专属折扣"],
        "valid_until": "2027-01-23"
    }

# 创建客户服务 Agent
agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history, get_current_promotions]
)

In [32]:
response_stream = agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "查询客户 id 为 cust1234的完整信息, 历史订单和可用优惠"
            }
        ]
    },
    stream_mode= "values"
)

for chunk in response_stream:
    rprint(chunk)
    print("-"*50)

{
    'messages': [
        HumanMessage(
            content='查询客户 id 为 cust1234的完整信息, 历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='6e7f65ea-8406-4730-87c9-7fd9bf072375'
        )
    ]
}

--------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户 id 为 cust1234的完整信息, 历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='6e7f65ea-8406-4730-87c9-7fd9bf072375'
        ),
        AIMessage(
            content='很抱歉，我目前无法查询客户的完整信息、历史订单和可用优惠。\n\n我当前可用的工具只有以下两个功能
：\n\n1. **生成销售报告** - 查看整体销售情况\n2. **生成库存报告** - 查看库存状况\n\n这两个工具都无法针对特定客户 
ID（如 cust1234）进行查询。\n\n如果您需要，我可以为您：\n- 生成销售报告（可能包含订单相关的整体信息）\n- 
生成库存报告\n\n或者，如果您需要查询特定客户的详细信息，建议通过客服系统或 CRM 
系统进行操作。请问您需要我生成上述哪份报告吗？',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 265,
                    'prompt_tokens': 185,
                    'total_tokens': 450,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 134,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3',
                'system_fingerprint': None,
                'id': '202611171343560518ed6dfa934432',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a1e808-8a94-7bb2-a22c-c9299d16d05f-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 185,
                'output_tokens': 265,
                'total_tokens': 450,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 134}
            }
        )
    ]
}

--------------------------------------------------


### 2. updates 输出模式

这种模式就是默认模式, 该模式中, 每个步骤执行后, 只增量更新状态中发生变化的内容, 用于监控 Agent 执行进度, 例如观察 Agent 决定调用工具, 工具执行结果等步骤

In [16]:
response_stream = agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "查询客户 id 为 cust1234的完整信息, 历史订单和可用优惠"
            }
        ]
    },
    stream_mode= "updates"
)

for chunk in response_stream:
    rprint(chunk)
    print("-"*50)

{
    'model': {
        'messages': [
            AIMessage(
                content='我将为您查询客户 cust1234 的完整信息、历史订单以及当前可用优惠。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 117,
                        'prompt_tokens': 353,
                        'total_tokens': 470,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 63,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5.3',
                    'system_fingerprint': None,
                    'id': '2026111615470802fc763f8a42423b',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--01a1e352-fad4-7513-9c74-9bab9253f262-0',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_-7079991628810680853',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_-7079991628810680852',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_-7079991628810680851',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 353,
                    'output_tokens': 117,
                    'total_tokens': 470,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 63}
                }
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
                name='query_customer_data',
                id='e0e3c328-1aa5-4da8-b43d-bfca28aeb514',
                tool_call_id='call_-7079991628810680853'
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"total_orders": 15, "total_spent": 25800.0}',
                name='check_order_history',
                id='46c777fa-6ab7-4fdf-b001-817e2a6559aa',
                tool_call_id='call_-7079991628810680852'
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-23"}',
                name='get_current_promotions',
                id='752110b9-aef8-41c1-992e-c3ec07229262',
                tool_call_id='call_-7079991628810680851'
            )
        ]
    }
}

--------------------------------------------------


{
    'model': {
        'messages': [
            AIMessage(
                content='已为您查询到客户 cust1234 的全部信息，以下是详细内容：\n\n## 👤 客户基本信息\n| 项目 | 
详情 |\n|------|------|\n| 姓名 | 张三 |\n| 会员等级 | VIP |\n| 加入日期 | 2023-01-15 |\n\n## 📦 历史订单情况\n- 
**总订单数**：15 单\n- **累计消费金额**：¥25,800.00\n\n## 🎁 当前可用优惠\n- 老用户优惠\n- 会员专属折扣\n- 
**有效期至**：2027-01-23\n\n**小结**：张三是VIP会员，累计消费近2.6万元，目前可享受两项优惠活动（老用户优惠 + 
会员专属折扣），优惠有效期至2027年1月23日。如需了解更多详情，请随时告知！',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 203,
                        'prompt_tokens': 485,
                        'total_tokens': 688,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 20,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5.3',
                    'system_fingerprint': None,
                    'id': '2026111615471213d04e579a6c4b98',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a1e353-0a81-7e21-b5ff-f8456501d6de-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 485,
                    'output_tokens': 203,
                    'total_tokens': 688,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 20}
                }
            )
        ]
    }
}

--------------------------------------------------


### 3. messages 输出模式

该模式中会输出流式返回的 Token 以及相关的元数据(如: 来自哪个节点), 可以用在实现类似 ChatGPT 的打字效果场景, 为聊天机器人等交互应用提供最佳的实现体验

In [20]:
response_stream = agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "查询客户 id 为 cust1234的完整信息, 历史订单和可用优惠"
            }
        ]
    },
    stream_mode= "messages"
)

for chunk in response_stream:
    # rprint(chunk)
    # print("-"*50)
    rprint(chunk[0].content, end="", flush=True)

我将

为您

查询

客户

cust

123

... (164 outputs left out)

提升

客户

满意度

。

### 4. tasks 输出模式

该模式会输出当前 task 任务开始和结束的时间, 包含任务的结束和错误信息, 改模式用于监控任务的生命周期

In [22]:
response_stream = agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "查询客户 id 为 cust1234的完整信息, 历史订单和可用优惠"
            }
        ]
    },
    stream_mode= "tasks"
)

for chunk in response_stream:
    rprint(chunk)
    print("-"*50)

{
    'id': '2ceb8c7d-b8e1-6993-ab4a-2e2f594a0b8d',
    'name': 'model',
    'input': {
        'messages': [
            HumanMessage(
                content='查询客户 id 为 cust1234的完整信息, 历史订单和可用优惠',
                additional_kwargs={},
                response_metadata={},
                id='13f72992-7189-4e8f-8835-4236ca451197'
            )
        ]
    },
    'triggers': ('branch:to:model',)
}

--------------------------------------------------


{
    'id': '2ceb8c7d-b8e1-6993-ab4a-2e2f594a0b8d',
    'name': 'model',
    'error': None,
    'result': {
        'messages': [
            AIMessage(
                content='我将为您查询客户 cust1234 的基本信息、历史订单以及当前可用的促销活动。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 138,
                        'prompt_tokens': 353,
                        'total_tokens': 491,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 83,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5.3',
                    'system_fingerprint': None,
                    'id': '20261116155042f41a4b263ed24b1e',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--01a1e356-3e0a-7330-a34d-03e6fd8093d1-0',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_-7079998260240182221',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_-7079998260240182220',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_-7079998260240182219',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 353,
                    'output_tokens': 138,
                    'total_tokens': 491,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 83}
                }
            )
        ]
    },
    'interrupts': []
}

--------------------------------------------------


{
    'id': '48416ca4-760c-9ba1-e051-c115835949e6',
    'name': 'tools',
    'input': {
        '__type': 'tool_call_with_context',
        'tool_call': {
            'name': 'query_customer_data',
            'args': {'customer_id': 'cust1234'},
            'id': 'call_-7079998260240182221',
            'type': 'tool_call'
        },
        'state': {
            'messages': [
                HumanMessage(
                    content='查询客户 id 为 cust1234的完整信息, 历史订单和可用优惠',
                    additional_kwargs={},
                    response_metadata={},
                    id='13f72992-7189-4e8f-8835-4236ca451197'
                ),
                AIMessage(
                    content='我将为您查询客户 cust1234 的基本信息、历史订单以及当前可用的促销活动。',
                    additional_kwargs={'refusal': None},
                    response_metadata={
                        'token_usage': {
                            'completion_tokens': 138,
                            'prompt_tokens': 353,
                            'total_tokens': 491,
                            'completion_tokens_details': {
                                'accepted_prediction_tokens': None,
                                'audio_tokens': None,
                                'reasoning_tokens': 83,
                                'rejected_prediction_tokens': None
                            },
                            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                        },
                        'model_provider': 'openai',
                        'model_name': 'glm-5.3',
                        'system_fingerprint': None,
                        'id': '20261116155042f41a4b263ed24b1e',
                        'finish_reason': 'tool_calls',
                        'logprobs': None
                    },
                    id='lc_run--01a1e356-3e0a-7330-a34d-03e6fd8093d1-0',
                    tool_calls=[
                        {
                            'name': 'query_customer_data',
                            'args': {'customer_id': 'cust1234'},
                            'id': 'call_-7079998260240182221',
                            'type': 'tool_call'
                        },
                        {
                            'name': 'check_order_history',
                            'args': {'customer_id': 'cust1234'},
                            'id': 'call_-7079998260240182220',
                            'type': 'tool_call'
                        },
                        {
                            'name': 'get_current_promotions',
                            'args': {},
                            'id': 'call_-7079998260240182219',
                            'type': 'tool_call'
                        }
                    ],
                    invalid_tool_calls=[],
                    usage_metadata={
                        'input_tokens': 353,
                        'output_tokens': 138,
                        'total_tokens': 491,
                        'input_token_details': {'cache_read': 0},
                        'output_token_details': {'reasoning': 83}
                    }
                )
            ]
        }
    },
    'triggers': ('__pregel_push',)
}

--------------------------------------------------


... (10 outputs left out)

{
    'id': 'd1f63414-4100-a974-b376-a968495fdf4e',
    'name': 'model',
    'input': {
        'messages': [
            HumanMessage(
                content='查询客户 id 为 cust1234的完整信息, 历史订单和可用优惠',
                additional_kwargs={},
                response_metadata={},
                id='13f72992-7189-4e8f-8835-4236ca451197'
            ),
            AIMessage(
                content='我将为您查询客户 cust1234 的基本信息、历史订单以及当前可用的促销活动。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 138,
                        'prompt_tokens': 353,
                        'total_tokens': 491,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 83,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5.3',
                    'system_fingerprint': None,
                    'id': '20261116155042f41a4b263ed24b1e',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--01a1e356-3e0a-7330-a34d-03e6fd8093d1-0',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_-7079998260240182221',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_-7079998260240182220',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_-7079998260240182219',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 353,
                    'output_tokens': 138,
                    'total_tokens': 491,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 83}
                }
            ),
            ToolMessage(
                content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
                name='query_customer_data',
                id='61b6f250-65cd-474b-beba-a6c02c8e594a',
                tool_call_id='call_-7079998260240182221'
            ),
            ToolMessage(
                content='{"total_orders": 15, "total_spent": 25800.0}',
                name='check_order_history',
                id='86a81c11-493a-41f1-b385-f7ff9cee3425',
                tool_call_id='call_-7079998260240182220'
            ),
            ToolMessage(
                content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-23"}',
                name='get_current_promotions',
                id='717fba7a-d01b-4a8e-9864-faa0d8e2da0e',
                tool_call_id='call_-7079998260240182219'
            )
        ]
    },
    'triggers': ('branch:to:model',)
}

--------------------------------------------------


{
    'id': 'd1f63414-4100-a974-b376-a968495fdf4e',
    'name': 'model',
    'error': None,
    'result': {
        'messages': [
            AIMessage(
                content='已为您查询到客户 **cust1234** 的完整信息，汇总如下：\n\n## 📋 客户基本信息\n- 
**姓名**：张三\n- **会员等级**：VIP\n- **加入日期**：2023年1月15日\n\n## 🛒 历史订单信息\n- **总订单数**：15 单\n- 
**累计消费金额**：¥25,800.00\n\n## 🎁 当前可用优惠活动\n- **老用户优惠**\n- **会员专属折扣**\n- 
**有效期至**：2027年1月23日\n\n**小结**：张三是一位 VIP 会员，加入至今已有较长时间，消费能力较强（平均每单约 
¥1,720），目前可享受两项专属优惠活动，且优惠有效期较长，建议提醒客户及时使用。\n\n如需了解更多详情，请随时告诉我！'
,
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 205,
                        'prompt_tokens': 486,
                        'total_tokens': 691,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 19,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 320}
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5.3',
                    'system_fingerprint': None,
                    'id': '20261116155048552ea7a3485e4f94',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a1e356-5336-7622-88fa-f789fa09a803-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 486,
                    'output_tokens': 205,
                    'total_tokens': 691,
                    'input_token_details': {'cache_read': 320},
                    'output_token_details': {'reasoning': 19}
                }
            )
        ]
    },
    'interrupts': []
}

--------------------------------------------------


### 5. debug 输出模式

In [23]:
response_stream = agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "查询客户 id 为 cust1234的完整信息, 历史订单和可用优惠"
            }
        ]
    },
    stream_mode= "debug"
)

for chunk in response_stream:
    rprint(chunk)
    print("-"*50)

{
    'step': 1,
    'timestamp': '2026-11-16T07:53:12.211244+00:00',
    'type': 'task',
    'payload': {
        'id': 'd03a5c66-98af-8959-6585-a4545a136f9d',
        'name': 'model',
        'input': {
            'messages': [
                HumanMessage(
                    content='查询客户 id 为 cust1234的完整信息, 历史订单和可用优惠',
                    additional_kwargs={},
                    response_metadata={},
                    id='97c32dcc-9377-466c-8c86-86b2ee367c9d'
                )
            ]
        },
        'triggers': ('branch:to:model',)
    }
}

--------------------------------------------------


{
    'step': 1,
    'timestamp': '2026-11-16T07:53:16.861034+00:00',
    'type': 'task_result',
    'payload': {
        'id': 'd03a5c66-98af-8959-6585-a4545a136f9d',
        'name': 'model',
        'error': None,
        'result': {
            'messages': [
                AIMessage(
                    content='我将为您查询客户 cust1234 的完整信息、历史订单以及当前可用优惠。',
                    additional_kwargs={'refusal': None},
                    response_metadata={
                        'token_usage': {
                            'completion_tokens': 147,
                            'prompt_tokens': 353,
                            'total_tokens': 500,
                            'completion_tokens_details': {
                                'accepted_prediction_tokens': None,
                                'audio_tokens': None,
                                'reasoning_tokens': 93,
                                'rejected_prediction_tokens': None
                            },
                            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                        },
                        'model_provider': 'openai',
                        'model_name': 'glm-5.3',
                        'system_fingerprint': None,
                        'id': '2026111615531240d635b51d504479',
                        'finish_reason': 'tool_calls',
                        'logprobs': None
                    },
                    id='lc_run--01a1e358-8723-7503-ae76-21c3c1d28101-0',
                    tool_calls=[
                        {
                            'name': 'query_customer_data',
                            'args': {'customer_id': 'cust1234'},
                            'id': 'call_-7079993965272888965',
                            'type': 'tool_call'
                        },
                        {
                            'name': 'check_order_history',
                            'args': {'customer_id': 'cust1234'},
                            'id': 'call_-7079993965272888964',
                            'type': 'tool_call'
                        },
                        {
                            'name': 'get_current_promotions',
                            'args': {},
                            'id': 'call_-7079993965272888963',
                            'type': 'tool_call'
                        }
                    ],
                    invalid_tool_calls=[],
                    usage_metadata={
                        'input_tokens': 353,
                        'output_tokens': 147,
                        'total_tokens': 500,
                        'input_token_details': {'cache_read': 0},
                        'output_token_details': {'reasoning': 93}
                    }
                )
            ]
        },
        'interrupts': []
    }
}

--------------------------------------------------


{
    'step': 2,
    'timestamp': '2026-11-16T07:53:16.861293+00:00',
    'type': 'task',
    'payload': {
        'id': '6f0bacda-a187-44fb-9db5-22643fc855e7',
        'name': 'tools',
        'input': {
            '__type': 'tool_call_with_context',
            'tool_call': {
                'name': 'query_customer_data',
                'args': {'customer_id': 'cust1234'},
                'id': 'call_-7079993965272888965',
                'type': 'tool_call'
            },
            'state': {
                'messages': [
                    HumanMessage(
                        content='查询客户 id 为 cust1234的完整信息, 历史订单和可用优惠',
                        additional_kwargs={},
                        response_metadata={},
                        id='97c32dcc-9377-466c-8c86-86b2ee367c9d'
                    ),
                    AIMessage(
                        content='我将为您查询客户 cust1234 的完整信息、历史订单以及当前可用优惠。',
                        additional_kwargs={'refusal': None},
                        response_metadata={
                            'token_usage': {
                                'completion_tokens': 147,
                                'prompt_tokens': 353,
                                'total_tokens': 500,
                                'completion_tokens_details': {
                                    'accepted_prediction_tokens': None,
                                    'audio_tokens': None,
                                    'reasoning_tokens': 93,
                                    'rejected_prediction_tokens': None
                                },
                                'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                            },
                            'model_provider': 'openai',
                            'model_name': 'glm-5.3',
                            'system_fingerprint': None,
                            'id': '2026111615531240d635b51d504479',
                            'finish_reason': 'tool_calls',
                            'logprobs': None
                        },
                        id='lc_run--01a1e358-8723-7503-ae76-21c3c1d28101-0',
                        tool_calls=[
                            {
                                'name': 'query_customer_data',
                                'args': {'customer_id': 'cust1234'},
                                'id': 'call_-7079993965272888965',
                                'type': 'tool_call'
                            },
                            {
                                'name': 'check_order_history',
                                'args': {'customer_id': 'cust1234'},
                                'id': 'call_-7079993965272888964',
                                'type': 'tool_call'
                            },
                            {
                                'name': 'get_current_promotions',
                                'args': {},
                                'id': 'call_-7079993965272888963',
                                'type': 'tool_call'
                            }
                        ],
                        invalid_tool_calls=[],
                        usage_metadata={
                            'input_tokens': 353,
                            'output_tokens': 147,
                            'total_tokens': 500,
                            'input_token_details': {'cache_read': 0},
                            'output_token_details': {'reasoning': 93}
                        }
                    )
                ]
            }
        },
        'triggers': ('__pregel_push',)
    }
}

--------------------------------------------------


... (10 outputs left out)

{
    'step': 3,
    'timestamp': '2026-11-16T07:53:16.916847+00:00',
    'type': 'task',
    'payload': {
        'id': '2bf00761-d42e-bb90-15e8-09dca8b463b2',
        'name': 'model',
        'input': {
            'messages': [
                HumanMessage(
                    content='查询客户 id 为 cust1234的完整信息, 历史订单和可用优惠',
                    additional_kwargs={},
                    response_metadata={},
                    id='97c32dcc-9377-466c-8c86-86b2ee367c9d'
                ),
                AIMessage(
                    content='我将为您查询客户 cust1234 的完整信息、历史订单以及当前可用优惠。',
                    additional_kwargs={'refusal': None},
                    response_metadata={
                        'token_usage': {
                            'completion_tokens': 147,
                            'prompt_tokens': 353,
                            'total_tokens': 500,
                            'completion_tokens_details': {
                                'accepted_prediction_tokens': None,
                                'audio_tokens': None,
                                'reasoning_tokens': 93,
                                'rejected_prediction_tokens': None
                            },
                            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                        },
                        'model_provider': 'openai',
                        'model_name': 'glm-5.3',
                        'system_fingerprint': None,
                        'id': '2026111615531240d635b51d504479',
                        'finish_reason': 'tool_calls',
                        'logprobs': None
                    },
                    id='lc_run--01a1e358-8723-7503-ae76-21c3c1d28101-0',
                    tool_calls=[
                        {
                            'name': 'query_customer_data',
                            'args': {'customer_id': 'cust1234'},
                            'id': 'call_-7079993965272888965',
                            'type': 'tool_call'
                        },
                        {
                            'name': 'check_order_history',
                            'args': {'customer_id': 'cust1234'},
                            'id': 'call_-7079993965272888964',
                            'type': 'tool_call'
                        },
                        {
                            'name': 'get_current_promotions',
                            'args': {},
                            'id': 'call_-7079993965272888963',
                            'type': 'tool_call'
                        }
                    ],
                    invalid_tool_calls=[],
                    usage_metadata={
                        'input_tokens': 353,
                        'output_tokens': 147,
                        'total_tokens': 500,
                        'input_token_details': {'cache_read': 0},
                        'output_token_details': {'reasoning': 93}
                    }
                ),
                ToolMessage(
                    content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
                    name='query_customer_data',
                    id='a55a9400-48f6-47ba-a7f9-9a3f25322be9',
                    tool_call_id='call_-7079993965272888965'
                ),
                ToolMessage(
                    content='{"total_orders": 15, "total_spent": 25800.0}',
                    name='check_order_history',
                    id='6a2b476a-6ab0-483a-ac2e-8a2cb577251f',
                    tool_call_id='call_-7079993965272888964'
                ),
                ToolMessage(
                    content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-23"}',
                    name='get_current_promotions',
                    id='40ac49d0-8439-4957-b7ee-a6f58d27b1d0',
                    tool_call_id='call_-7079993965272888963'
                )
            ]
        },
        'triggers': 

--------------------------------------------------


{
    'step': 3,
    'timestamp': '2026-11-16T07:53:22.640151+00:00',
    'type': 'task_result',
    'payload': {
        'id': '2bf00761-d42e-bb90-15e8-09dca8b463b2',
        'name': 'model',
        'error': None,
        'result': {
            'messages': [
                AIMessage(
                    content='以下是客户 **cust1234** 的完整查询结果:\n\n## 👤 客户基本信息\n| 项目 | 详情 
|\n|------|------|\n| 姓名 | 张三 |\n| 会员等级 | VIP |\n| 入会日期 | 2023-01-15 |\n\n## 📦 历史订单\n- 
**总订单数:** 15 笔\n- **累计消费:** ¥25,800.00\n- **平均单笔消费:** 约 ¥1,720.00\n\n## 🎁 
当前可用优惠\n以下促销活动有效期至 **2027-01-23**:\n1. 老用户优惠\n2. 会员专属折扣\n\n💡 **小提示:** 该客户为 VIP 
会员且消费活跃(15笔订单,累计消费2.58万元),建议向其推荐"会员专属折扣"活动,进一步提升客户满意度和复购率。\n\n如需了解
更详细的订单明细或其他信息,请随时告诉我!',
                    additional_kwargs={'refusal': None},
                    response_metadata={
                        'token_usage': {
                            'completion_tokens': 220,
                            'prompt_tokens': 485,
                            'total_tokens': 705,
                            'completion_tokens_details': {
                                'accepted_prediction_tokens': None,
                                'audio_tokens': None,
                                'reasoning_tokens': 19,
                                'rejected_prediction_tokens': None
                            },
                            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                        },
                        'model_provider': 'openai',
                        'model_name': 'glm-5.3',
                        'system_fingerprint': None,
                        'id': '20261116155317cf19092f18bb4aa4',
                        'finish_reason': 'stop',
                        'logprobs': None
                    },
                    id='lc_run--01a1e358-998d-7da0-9b88-6066f0fcce59-0',
                    tool_calls=[],
                    invalid_tool_calls=[],
                    usage_metadata={
                        'input_tokens': 485,
                        'output_tokens': 220,
                        'total_tokens': 705,
                        'input_token_details': {'cache_read': 0},
                        'output_token_details': {'reasoning': 19}
                    }
                )
            ]
        },
        'interrupts': []
    }
}

--------------------------------------------------


### 6. checkpoints 输出模式

该模式中, 每当检查点(checkpoint)被创建时会触发输出, 输出包含检查点中的状态, 用于需要状态持久化, 工作流恢复或分布式执行跟踪的高级场景

In [28]:
from langgraph.checkpoint.memory import InMemorySaver

# 1. 创建内存检查点存储
checkpointer = InMemorySaver()

# 2. 创建 Agent
agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history, get_current_promotions],
    checkpointer=checkpointer # 启用检查点
)

# 3. 创建唯一的会话ID
config = {"configurable": {"thread_id": "session01"}}

# 4. 调用 Agent
checkpointer_count = 0

# 使用 checkpoints 模式进行流式监控
response_stream = agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "查询客户 id 为 cust1234的完整信息, 历史订单和可用优惠"
            }
        ]
    },
    config=config,
    stream_mode= "checkpoints"
)

for chunk in response_stream:
    rprint(chunk)
    print("-"*50)

{
    'config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1e19ed-d393-6fae-bfff-082704373406'
        }
    },
    'parent_config': None,
    'values': {'messages': []},
    'metadata': {'source': 'input', 'step': -1, 'parents': {}},
    'next': ['__start__'],
    'tasks': [
        {'id': 'fbe54f76-15ff-18fb-fa21-235274018d40', 'name': '__start__', 'interrupts': (), 'state': None}
    ]
}

--------------------------------------------------


{
    'config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1e19ed-d3a7-6928-8000-774ff2aa2534'
        }
    },
    'parent_config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1e19ed-d393-6fae-bfff-082704373406'
        }
    },
    'values': {
        'messages': [
            HumanMessage(
                content='查询客户 id 为 cust1234的完整信息, 历史订单和可用优惠',
                additional_kwargs={},
                response_metadata={},
                id='9dbecd2c-5ab6-41eb-92f5-4dbd9ee8eeea'
            )
        ]
    },
    'metadata': {'source': 'loop', 'step': 0, 'parents': {}},
    'next': ['model'],
    'tasks': [{'id': '172d35c1-9a58-8db6-80a0-afe4fa50f1a1', 'name': 'model', 'interrupts': (), 'state': None}]
}

--------------------------------------------------


{
    'config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1e19ee-020b-6bca-8001-9ebaa28dc868'
        }
    },
    'parent_config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1e19ed-d3a7-6928-8000-774ff2aa2534'
        }
    },
    'values': {
        'messages': [
            HumanMessage(
                content='查询客户 id 为 cust1234的完整信息, 历史订单和可用优惠',
                additional_kwargs={},
                response_metadata={},
                id='9dbecd2c-5ab6-41eb-92f5-4dbd9ee8eeea'
            ),
            AIMessage(
                content='我来为您查询客户 cust1234 的完整信息、历史订单和当前可用优惠。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 161,
                        'prompt_tokens': 353,
                        'total_tokens': 514,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 107,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5.3',
                    'system_fingerprint': None,
                    'id': '202611161719462152c27ac2c74b0c',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--01a1e3a7-c687-7ca0-a3e2-3f72f00a99c0-0',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_-7079991182134080735',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_-7079991182134080734',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_-7079991182134080733',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 353,
                    'output_tokens': 161,
                    'total_tokens': 514,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 107}
                }
            )
        ]
    },
    'metadata': {'source': 'loop', 'step': 1, 'parents': {}},
    'next': ['tools', 'tools', 'tools'],
    'tasks': [
        {'id': 'b6fbd3a6-6528-e65a-b9fa-f3fa8dd0614e', 'name': 'tools', 'interrupts': (), 'state': None},
        {'id': '5fc66491-08ed-2f5e-8951-1548ded01207', 'name': 'tools', 'interrupts': (), 'state': None},
        {'id': '1407a2ac-b83c-b522-326c-daacbd7bd00e', 'name': 'tools', 'interrupts': (), 'state': None}
    ]
}

--------------------------------------------------


{
    'config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1e19ee-0244-6d9e-8002-f0d22293bc5c'
        }
    },
    'parent_config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1e19ee-020b-6bca-8001-9ebaa28dc868'
        }
    },
    'values': {
        'messages': [
            HumanMessage(
                content='查询客户 id 为 cust1234的完整信息, 历史订单和可用优惠',
                additional_kwargs={},
                response_metadata={},
                id='9dbecd2c-5ab6-41eb-92f5-4dbd9ee8eeea'
            ),
            AIMessage(
                content='我来为您查询客户 cust1234 的完整信息、历史订单和当前可用优惠。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 161,
                        'prompt_tokens': 353,
                        'total_tokens': 514,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 107,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5.3',
                    'system_fingerprint': None,
                    'id': '202611161719462152c27ac2c74b0c',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--01a1e3a7-c687-7ca0-a3e2-3f72f00a99c0-0',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_-7079991182134080735',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_-7079991182134080734',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_-7079991182134080733',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 353,
                    'output_tokens': 161,
                    'total_tokens': 514,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 107}
                }
            ),
            ToolMessage(
                content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
                name='query_customer_data',
                id='6ec3d7be-5156-4557-9acf-4772a8a292b5',
                tool_call_id='call_-7079991182134080735'
            ),
            ToolMessage(
                content='{"total_orders": 15, "total_spent": 25800.0}',
                name='check_order_history',
                id='c0b549d0-5ad2-49c9-bb9d-8479a620e93b',
                tool_call_id='call_-7079991182134080734'
            ),
            ToolMessage(
                content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-23"}',
                name='get_current_promotions',
                id='8ed45de2-36fc-483a-b6bb-8be7bfefb001',
                tool_call_id='call_-7079991182134080733'
            )
        ]
    },
    'metadata': {'source': 'loop', 'step': 2, 'parents': {}},
    'next': ['model'],
    'tasks': [{'id': 'a2f34c16-e048-28

--------------------------------------------------


{
    'config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1e19ee-372c-6304-8003-ce8e3d134a6d'
        }
    },
    'parent_config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1e19ee-0244-6d9e-8002-f0d22293bc5c'
        }
    },
    'values': {
        'messages': [
            HumanMessage(
                content='查询客户 id 为 cust1234的完整信息, 历史订单和可用优惠',
                additional_kwargs={},
                response_metadata={},
                id='9dbecd2c-5ab6-41eb-92f5-4dbd9ee8eeea'
            ),
            AIMessage(
                content='我来为您查询客户 cust1234 的完整信息、历史订单和当前可用优惠。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 161,
                        'prompt_tokens': 353,
                        'total_tokens': 514,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 107,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5.3',
                    'system_fingerprint': None,
                    'id': '202611161719462152c27ac2c74b0c',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--01a1e3a7-c687-7ca0-a3e2-3f72f00a99c0-0',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_-7079991182134080735',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'cust1234'},
                        'id': 'call_-7079991182134080734',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_-7079991182134080733',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 353,
                    'output_tokens': 161,
                    'total_tokens': 514,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 107}
                }
            ),
            ToolMessage(
                content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
                name='query_customer_data',
                id='6ec3d7be-5156-4557-9acf-4772a8a292b5',
                tool_call_id='call_-7079991182134080735'
            ),
            ToolMessage(
                content='{"total_orders": 15, "total_spent": 25800.0}',
                name='check_order_history',
                id='c0b549d0-5ad2-49c9-bb9d-8479a620e93b',
                tool_call_id='call_-7079991182134080734'
            ),
            ToolMessage(
                content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-23"}',
                name='get_current_promotions',
                id='8ed45de2-36fc-483a-b6bb-8be7bfefb001',
                tool_call_id='call_-7079991182134080733'
            ),
            AIMessage(
                content='已为您查询到客户 **cust1234** 的全部信息：\n\n## 👤 客户基本信息\n- **姓名**：张三\n- 
**等级**：VIP\n- **加入日期**：2023-01

--------------------------------------------------


### 7. costom 模式

开发者通过 get_stream_writer 在工具或节点内部自定义发送的数据, 用于输出业务逻辑相关的进度信息(如: 已处理10/100条记录), 自定义日志或指标
举例: 生成销售报告和库存报告 Agent

In [30]:
import time
from langgraph.config import get_stream_writer

@tool
def generate_sales_report() -> str:
    """生成销售报告"""
    writer = get_stream_writer()

    writer({"type": "生成销售报告", "message": "开始生成销售报告"})

    # 模拟数据处理
    for i in range(1, 4):
        time.sleep(0.5)
        writer({"type": "生成销售报告", "message": f"生成销售报告进度百分比: {i*25}%"})

    writer({"type": "生成销售报告", "message": "报告生成完成"})
    return f"销售报告: 总收入150万元, 同比增长12%"

@tool
def generate_inventory_report() -> str:
    """生成库存报告"""
    writer = get_stream_writer()
    writer("开始库存分析...")
    time.sleep(0.5)
    writer("检查当前库存量...")
    time.sleep(0.5)
    writer("生成库存报告")
    return "当前库存量为 10000 件, 库存充足, 无异常"

# 创建报告生成 agent
agent = create_agent(
    model=model,
    tools=[generate_sales_report, generate_inventory_report]
)

response_stream = agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "查询客户 id 为 cust1234的完整信息, 历史订单和可用优惠"
            }
        ]
    },
    stream_mode= "custom"
)

for chunk in response_stream:
    rprint(chunk)
    print("-"*50)